# Encoding Models with FastText Embeddings

This notebook adapts a standard encoding model pipeline to link static **FastText word embeddings** with ECoG neural data.
This pipeline is contained in the file static_encoding.py, and called here.

**Pipeline Overview:**
1.  **Load Features:** Load transcript and pre-computed FastText embeddings from a CSV.
2.  **Load Brain Data:** Load ECoG data using `mne`, epoch around word onsets, and downsample.
3.  **Alignment:** Sync the embedding matrix (X) with valid brain data epochs (Y).
4.  **Modeling:** Train a Ridge Regression model using `Himalaya` (with GPU acceleration if available) to predict brain activity from word vectors.

**Input Requirements:**
* **Transcript CSV:** A file containing columns `word`, `start`, `end`, and `embedding` (150-d float32 array).
* **Brain Data:** Preprocessed `.fif` file (MNE format).

# LOOP OVER ALL OPTIONS

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

import sys
sys.path.append("../scripts")

from static_encoding import process_embeddings

from nilearn.plotting import plot_markers

import io
from contextlib import redirect_stdout
import warnings
import os

import mne
from mne_bids import BIDSPath

In [2]:
def parse_array(s):
    if not isinstance(s, str): return s
    s = s.replace("\n", " ").strip("[]").strip()
    return np.array([float(x) for x in s.split() if x], dtype=np.float32)

In [3]:
embedding_filename = r"C:\Users\mayat\OneDrive\Desktop\lab\Language-Project-main\data\processed\podcast_trilingual_embeddings.csv"
embedding_df = pd.read_csv(embedding_filename, converters={
            "en_embedding": parse_array,
            "he_embedding": parse_array,
            "ar_embedding": parse_array})

In [4]:
## CONFIG ##
freq = 64
tmin, tmax = -2.0, 2.0
# use_PCA = True
use_PCA = False
PCA_dim = 150
datapath = r"C:\Users\mayat\ds005574-download\derivatives\ecogprep"

outpath = rf"C:\Users\mayat\OneDrive\Desktop\lab\Language-Project-main\data\processed\encoding_results_{freq}Hz_({tmin},{tmax})\\"
##################

subjects = [f"{i:02d}" for i in range(1, 10)]
mode_list = ["en", "he", "ar", "noise", "en+he", "en+ar", "en+noise", "all"]
if use_PCA is False:
    mode_list = ["en+he", "en+ar", "en+noise", "all"]

mode_list

['en+he', 'en+ar', 'en+noise', 'all']

In [ ]:
#os.path.join(datapath, file_path)

'./subject_data/sub-01_task-podcast_desc-highgamma_ieeg.fif'

In [ ]:
os.makedirs(outpath, exist_ok=True)

for subj in subjects:
    file_path = os.path.join(
        datapath,
        f"sub-{subj}",
        "ieeg",
        f"sub-{subj}_task-podcast_desc-highgamma_ieeg.fif"
    )

    print("Loading:", file_path)

    raw = mne.io.read_raw_fif(file_path, verbose=False)

    for language_mode in tqdm(mode_list, desc=f"Subject {subj}"):
        noise_mode = "over all embeds"

        # Process embeddings
        f = io.StringIO()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            with redirect_stdout(f):
                _, cv_scores = process_embeddings(
                    embedding_df=embedding_df,
                    raw=raw,
                    channel_names_regex="",
                    freq=freq,
                    tmin=tmin,
                    tmax=tmax,
                    language_mode=language_mode,
                    random_noise_mode=noise_mode,
                    use_PCA=use_PCA,
                    PCA_dim=PCA_dim
                )

        
        cv_score_filename = f"corrs subj={subj} - "
        time_title = f"Subject {subj}, correlation w/ time\n"
        time_filename = f"correlation time subj={subj} - "
        electrode_title = f"Subject {subj}, correlation w/ electrodes\n"
        electrode_filename = f"correlation electrodes subj={subj} - "
        
        filename_parts = []
        title_parts = []

        if language_mode == "all":
            filename_parts = ["English", "Hebrew", "Arabic"]
            title_parts = ["English", "Hebrew", "Arabic"]

        elif language_mode == "noise":  # i.e no actual language
            noise_suffix = f"Noise {noise_mode}"
            filename_parts.append(noise_suffix)
            title_parts.append(noise_suffix)

        else:  # not only noise
            if "en" in language_mode:
                filename_parts.append("English")
                title_parts.append("English")
            if "he" in language_mode:
                filename_parts.append("Hebrew")
                title_parts.append("Hebrew")
            if "ar" in language_mode:
                filename_parts.append("Arabic")
                title_parts.append("Arabic")
            if "noise" in language_mode:
                noise_suffix = f"noise {noise_mode}"
                filename_parts.append(noise_suffix)
                title_parts.append(noise_suffix)

        if use_PCA and ("+" in language_mode or language_mode == "all"):
            filename_parts.append("PCA")
            title_parts.append(f"\nAfter PCA down to {PCA_dim} dims")

        base_filename = "+".join(filename_parts)
        base_title = " + ".join(title_parts)

        cv_score_filename += base_filename
        time_filename += base_filename
        electrode_filename += base_filename
        time_title += base_title
        electrode_title += base_title
        
        # Save cv_scores
        np.save(outpath+cv_score_filename + ".npy", cv_scores)

        # Plot correlation over time
        lags = np.arange(tmin * 512, tmax * 512, (512 / freq)) / 512
        mean = cv_scores.mean((0, 1))
        err = cv_scores.std((0, 1)) / np.sqrt(np.prod(cv_scores.shape[:2]))
        
        fig, ax = plt.subplots()
        ax.plot(lags, mean, color='black')
        ax.fill_between(lags, mean - err, mean + err, alpha=0.1, color='black')
        ax.set_xlabel("lag (s)")
        ax.set_ylabel("encoding performance (r ± sem)")
        ax.axvline(0, c=(.9, .9, .9), ls="--")
        ax.axhline(0, c=(.9, .9, .9), ls="--")
        ax.set_ylim(-0.02, 0.05)
        
        
        ax.set_title(time_title)
        fig.savefig(f"{outpath+time_filename}.png", dpi=600, bbox_inches='tight')
        # fig.show()
        plt.close(fig)


        # Plot correlation over electrodes
        values = cv_scores.mean(0).max(-1)
        ch2loc = {ch['ch_name']: ch['loc'][:3] for ch in raw.info['chs']}
        coords = np.vstack([ch2loc[ch] for ch in raw.info['ch_names']]) * 1000

        order = values.argsort()
        lzr = plot_markers(values[order], coords[order],
                           node_size=30, display_mode='lzr',
                           node_vmin=0, node_vmax=0.28,
                           node_cmap='inferno_r', colorbar=True)
          

        lzr.title(electrode_title, size=10)
        lzr.savefig(f"{outpath+electrode_filename}.png", dpi=600, bbox_inches='tight')
        plt.close()

IndentationError: unexpected indent (2963795205.py, line 14)